# Galaxy Bias Analysis: P-Millennium GALFORM Snapshots

Investigate galaxy clustering bias by comparing galaxy and dark matter two-point correlation functions across P-Millennium merger tree subvolumes.

In [ ]:
# Ensure local package path is first
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)


# Now import libraries
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from utils.matplotlib_config import register_matplotlib_setconfig
from config import get_base_dir

# Import functions
from analysis.correlation import (
    compute_galaxy_bias,
    avg_galaxy_bias_over_subvolumes,
)

base_dir = get_base_dir()
register_matplotlib_setconfig(mpl)
mpl.setconfig()

# Plotting defaults
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 11

## Single Subvolume Example

In [ ]:
# Parameters for analysis
example_snapshot = 'iz207'
example_ivol = 180
snap_path = str(base_dir / example_snapshot)

# Define bin edges
rbins = np.logspace(np.log10(0.1), np.log10(50.0), 20)

# Optional: apply minimum halo mass cut
mhalo_min = 1e11  # Msun

result = compute_galaxy_bias(
    iz_path=snap_path,
    ivol=example_ivol,
    rbins=rbins,
    nthreads=4,
    centrals_only=False,  # All galaxies (centrals + satellites)
    mhalo_min=mhalo_min,
)
print(f"Redshift: {result.attrs.get('z'):.4f}")
print(f"N_galaxies (all): {result.attrs.get('ngal')}")
print(f"N_halos (DM centers): {result.attrs.get('nhalo')}")
print(f"Galaxies per halo: {result.attrs.get('ngal') / result.attrs.get('nhalo'):.2f}")

In [ ]:
r = result['r'].to_numpy()
xi_gal = result['xi_gal'].to_numpy()  # Already xi (not 1+xi)
xi_dm = result['xi_dm'].to_numpy()

# Compute bias correctly: b(r) = sqrt(xi_gal / xi_dm)
bias = np.full_like(xi_gal, np.nan)
mask = (xi_gal > 0) & (xi_dm > 0)
bias[mask] = np.sqrt(xi_gal[mask] / xi_dm[mask])

# Create summary table
print(f"\n{'r [Mpc/h]':>12}  {'ξ_gal(r)':>12}  {'ξ_DM(r)':>12}  {'bias':>8}")
print("-" * 50)
for i in range(len(r)):
    if np.isfinite(bias[i]):
        print(f"{r[i]:12.2f}  {xi_gal[i]:12.6f}  {xi_dm[i]:12.6f}  {bias[i]:8.3f}")

# Two-panel plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Correlation functions (only positive values for log-log)
ax = axes[0]
gal_mask = np.isfinite(xi_gal) & (xi_gal > 0)
dm_mask = np.isfinite(xi_dm) & (xi_dm > 0)
ax.loglog(r[gal_mask], xi_gal[gal_mask], 'o-', color='blue',
                label=f'All galaxies (N={result.attrs.get("ngal")})', markersize=6)
ax.loglog(r[dm_mask], xi_dm[dm_mask], 's-', color='darkred',
                label=f'DM halo centers (N={result.attrs.get("nhalo")})', markersize=6)
ax.set_xlabel('r [Mpc/h]', fontsize=12)
ax.set_ylabel('ξ(r)', fontsize=12)
ax.set_title(f'Two-Point Correlation Functions (z={result.attrs.get("z"):.2f})', fontsize=11)
ax.grid(True, alpha=0.3, which='both')
ax.legend(fontsize=10)

# Right: Galaxy bias
ax = axes[1]
ax.semilogx(r[mask], bias[mask], 'o-', color='green', markersize=6, linewidth=2)
ax.axhline(1, color='gray', linestyle='--', alpha=0.5, label='b=1 (unbiased)')
ax.set_xlabel('r [Mpc/h]', fontsize=12)
ax.set_ylabel('Galaxy bias b(r)', fontsize=12)
ax.set_title(f'Galaxy Bias (z={result.attrs.get("z"):.2f})', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

## Averaged Over Multiple Subvolumes

In [ ]:
# Fetch average galaxy bias data over multiple subvolumes
example_snapshot = 'iz207'
snap_path = str(base_dir / example_snapshot)
example_ivols = [40, 60, 100]  # Multiple subvolumes

avg_result = avg_galaxy_bias_over_subvolumes(
    iz_path=snap_path,
    ivols=example_ivols,
    rbins=rbins,
    nthreads=4,
    centrals_only=False,  # All galaxies
    mhalo_min=mhalo_min,
)

print(f"Redshift: {avg_result.attrs.get('z'):.4f}")
print(f"Subvolumes successful: {avg_result.attrs.get('n_used')}/{avg_result.attrs.get('n_requested')}")

In [ ]:
r = avg_result['r'].to_numpy()
xi_gal = avg_result['xi_gal'].to_numpy()
xi_dm = avg_result['xi_dm'].to_numpy()

# Compute bias correctly: b(r) = sqrt(xi_gal / xi_dm)
bias = np.full_like(xi_gal, np.nan)
mask = (xi_gal > 0) & (xi_dm > 0)
bias[mask] = np.sqrt(xi_gal[mask] / xi_dm[mask])
bias_std = avg_result['bias_std'].to_numpy()

# Create summary table
print(f"\n{'r [Mpc/h]':>12}  {'bias':>10}  {'std':>10}")
print("-" * 35)
for i in range(len(r)):
    if mask[i]:
        print(f"{r[i]:12.2f}  {bias[i]:10.3f}  {bias_std[i]:10.3f}")

# Plot with error bars
plt.figure(figsize=(10, 6))
plt.errorbar(r[mask], bias[mask], yerr=bias_std[mask], 
                fmt='o-', color='green', markersize=6, linewidth=2,
                capsize=4, label=f'Mean bias (N={avg_result.attrs.get("n_used")} subvolumes)')
plt.axhline(1, color='gray', linestyle='--', alpha=0.5, label='b=1 (unbiased)')
plt.xscale('log')
plt.xlabel('r [Mpc/h]', fontsize=12)
plt.ylabel('Galaxy bias b(r)', fontsize=12)
plt.title(f'Average Galaxy Bias (z={avg_result.attrs.get("z"):.2f})', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

## Redshift Evolution

In [ ]:
# Fetch galaxy bias data at multiple redshifts
iz_nums = [207]
example_ivol = 100

redshift_results = []
for iz_num in iz_nums:
    snap_path = str(base_dir / f'iz{iz_num}')
    try:
        res = compute_galaxy_bias(
            iz_path=snap_path,
            ivol=example_ivol,
            rbins=rbins,
            nthreads=4,
            centrals_only=False,  # All galaxies
            mhalo_min=mhalo_min,
        )
        
        z = res.attrs.get('z')
        ngal = res.attrs.get('ngal')
        nhalo = res.attrs.get('nhalo')
        redshift_results.append(res)
        print(f"iz{iz_num}: z={z:.3f}, N_gal={ngal}, N_halo={nhalo}")
    
    except Exception as e:
        print(e)

print(f"Successfully computed {len(redshift_results)} snapshot(s)")

In [ ]:
plt.figure(figsize=(10, 6))

colors = ['#d62728', '#1f77b4']
linestyles = ['-', '--']
markers = ['o', 's']

for i, res in enumerate(redshift_results):
    r = res["r"].to_numpy()
    xi_gal = res["xi_gal"].to_numpy()  # Already xi (not 1+xi)
    xi_dm = res["xi_dm"].to_numpy()
    
    # Compute bias correctly: b(r) = sqrt(xi_gal / xi_dm)
    bias = np.sqrt(xi_gal / xi_dm)
    mask = np.isfinite(bias)
    z = res.attrs.get("z")
    label = f"z={z:.2f}" if z is not None else "z=unknown"
    
    plt.semilogx(r[mask], bias[mask], linestyle=linestyles[i], marker=markers[i],
                    color=colors[i], label=label, markersize=6, linewidth=2.5, alpha=0.8)

plt.axhline(1, color='gray', linestyle=':', alpha=0.6, linewidth=1.5)
plt.xlabel('r [Mpc/h]', fontsize=12)
plt.ylabel('Galaxy bias b(r)', fontsize=12)
plt.title('Galaxy Bias: z=0 vs z≈0.5', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11, loc='best')
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()